# 03. 반 이닝 클립 정답 라벨 만들기

투구 기록(행)을 **반 이닝 단위 이벤트 시퀀스**로 바꾼다. 각 이벤트에는 그 직후의 B-S-O 상태를 붙인다.

| type | 의미 | 상태 변화 |
|---|---|---|
| `pitch` | 투구 (`result`: B/T/S/F/V/W/H, 사구는 `HBP`) | `apply_pitch` 규칙 |
| `auto_ball` / `auto_strike` | 투구 없이 볼/스트라이크 추가 (피치클락 위반) | B+1 / S+1 |
| `ibb` | 투구 없는 고의4구 (타석 시작 시) | 타석 종료 |
| `walk_award` | 타석 도중 투구 없이 볼넷 (자동 고의4구 등) | 타석 종료 |
| `runner_out` | 주루사로 이닝 종료 (타석 미완료) | 타석 종료 |

규칙
- 정답 카운트는 데이터 값을 그대로 쓴다. 데이터 카운트가 +1볼 / +1스트라이크로 튀면 `auto_*` 이벤트로 해석한다.
- 그 외 설명 안 되는 불일치가 있는 반 이닝은 통째로 제외한다. (2023 경기 첫 타석 오류 등)
- 타석이 끝나는 이벤트의 상태는 `0-0`, 아웃은 `post_outs`. 타석 도중 아웃 변화(도루 실패 등)는 타석 종료 시점에 반영된다.

In [17]:
# Google Drive 연결 (런타임이 새로 시작될 때마다 필요). 결과를 Drive에 저장해서 런타임이 끊겨도 남게 한다
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/kbo-listen")
DRIVE.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import json

import pandas as pd

# 재현성을 위해 데이터셋 커밋을 고정
REVISION = "6afc8af044e3bba5f326b688e8cb41d7ff7065ec"
BASE = f"https://huggingface.co/datasets/slothman3878/kbo_playbyplay/resolve/{REVISION}/v0"
SEASONS = [2023, 2024, 2025, 2026]

df = pd.concat(
    [pd.read_parquet(f"{BASE}/kbo_pbp_{y}.parquet").assign(season=y) for y in SEASONS],
    ignore_index=True,
)
# itertuples()는 '_'로 시작하는 컬럼명을 위치 이름(_27 등)으로 바꿔버리므로 미리 이름 변경
df = df.rename(columns={"_naver_pitch_name": "pitch_name_ko"})

print(df.shape)
print(df["season"].value_counts().sort_index())

(805960, 49)
season
2023    219934
2024    223277
2025    217970
2026    144779
Name: count, dtype: int64


In [2]:
# 02에서 검증한 상태머신 그대로
STRIKE_CODES = {"T", "S", "V", "W"}


def apply_pitch(balls, strikes, result):
    if result == "B":
        balls += 1
        return balls, strikes, balls == 4
    if result in STRIKE_CODES:
        strikes += 1
        return balls, strikes, strikes == 3
    if result == "F":
        return balls, min(strikes + 1, 2), False
    if result == "H":
        return balls, strikes, True
    raise ValueError(f"알 수 없는 투구 결과: {result}")

In [3]:
def num(x):
    """NaN → None, 나머지는 int (JSON 저장용)"""
    return None if pd.isna(x) else int(x)


def build_pa(rows):
    """타석 하나(투구 행 리스트) → (이벤트 리스트, 태그 집합, 정상 여부)"""
    first, last = rows[0], rows[-1]
    who = {"ab": int(first.at_bat_number), "pitcher": first.pitcher_name, "batter": first.batter_name}
    outs = int(first.outs_when_up)
    result = None if pd.isna(last.events) else last.events  # 결측이 None/NaN 어느 쪽으로 와도 처리
    events, tags = [], set()

    if first.pitch_number == 0:  # 투구 없는 고의4구
        events.append({"type": "ibb", **who})
        tags.add("ibb")
    else:
        balls, strikes, over = 0, 0, False
        for r in rows:
            if over:  # 상태머신상 끝난 타석에 투구가 더 있음
                return [], tags, False
            if (r.balls, r.strikes) == (balls + 1, strikes):
                events.append({"type": "auto_ball", **who, "b": balls + 1, "s": strikes, "o": outs})
                tags.add("pitch_clock")
            elif (r.balls, r.strikes) == (balls, strikes + 1):
                events.append({"type": "auto_strike", **who, "b": balls, "s": strikes + 1, "o": outs})
                tags.add("pitch_clock")
            elif (r.balls, r.strikes) != (balls, strikes):
                return [], tags, False  # 설명 안 되는 불일치
            balls, strikes, over = apply_pitch(r.balls, r.strikes, r.pitch_result)
            events.append({
                "type": "pitch", **who, "pitch": int(r.pitch_number), "result": r.pitch_result,
                "pitch_name": r.pitch_name_ko, "speed": num(r.release_speed_kmh),
                "b": balls, "s": strikes, "o": outs,
            })

        # 카운트만으로는 안 보이는 타석 종료 처리
        if result == "hit_by_pitch":
            events[-1]["result"] = "HBP"  # 데이터에는 B로 기록돼 있음
            tags.add("hbp")
        elif result == "walk" and not over:
            events.append({"type": "walk_award", **who})
            tags.add("walk_award")
        elif result == "strikeout" and not over:
            events.append({"type": "auto_strike", **who})
            tags.add("pitch_clock")
        elif result is None:
            if last.post_outs != 3:
                return [], tags, False
            events.append({"type": "runner_out", **who})
            tags.add("runner_out")

    # 타석 종료: 카운트 리셋, 아웃은 post_outs
    events[-1].update({"b": 0, "s": 0, "o": int(last.post_outs), "pa_end": result or "runner_out"})
    return events, tags, True

In [4]:
import random

HALF = ["game_pk", "inning", "inning_topbot"]

# 분할 (C안): 2023·2024 + 2025 절반 → train / 2025 나머지 절반 → dev / 2026 → test
# 같은 경기의 반 이닝들이 train과 dev에 섞이지 않도록 경기 단위로 나눈다
games_2025 = sorted(df.loc[df["season"] == 2025, "game_pk"].unique())
random.Random(42).shuffle(games_2025)
dev_games = set(games_2025[: len(games_2025) // 2])


def split_of(season, game):
    if season == 2026:
        return "test"
    if game in dev_games:
        return "dev"
    return "train"


clips, dropped = [], []

for (game, inning, topbot), half in df.groupby(HALF, sort=False):
    first = half.iloc[0]
    clip_id = f"{game}_{inning:02d}{'T' if topbot == 'top' else 'B'}"
    events, tags, ok = [], set(), True
    prev_outs = 0  # 반 이닝은 0아웃에서 시작

    for _, pa in half.groupby("at_bat_number", sort=False):
        rows = list(pa.itertuples(index=False))
        # 아웃 이음새: 이번 타석 시작 아웃 == 직전 타석 종료 아웃 (첫 타석은 0)
        if rows[0].outs_when_up != prev_outs:
            ok = False
            break
        pa_events, pa_tags, ok = build_pa(rows)
        if not ok:
            break
        events += pa_events
        tags |= pa_tags
        prev_outs = rows[-1].post_outs

    if not ok:
        dropped.append(clip_id)
        continue

    season = int(first.season)
    clips.append({
        "clip_id": clip_id,
        "split": split_of(season, game),
        "season": season,
        "game_pk": game,
        "inning": int(inning),
        "half": topbot,
        "away": first.away_team,
        "home": first.home_team,
        "score": {"away": int(first.away_score), "home": int(first.home_score)},
        "tags": sorted(tags),
        "n_pitches": sum(e["type"] == "pitch" for e in events),
        "events": events,
    })

print("클립:", len(clips), "/ 제외:", len(dropped))
print("제외 예시:", dropped[:10])

클립: 46787 / 제외: 42
제외 예시: ['20230609LGHH02023_09B', '20230827WOSS02023_01T', '20230913LTHT02023_01T', '20230915LGHH02023_01T', '20230915SSNC02023_01T', '20230921OBSS02023_01T', '20231004NCSK02023_01T', '20231004HTKT22023_01T', '20240404LTHH02024_05B', '20240404LTHH02024_07T']


In [9]:
# split·시즌별 클립 수와 특수 이벤트 태그 분포
all_tags = sorted({t for c in clips for t in c["tags"]})
meta = pd.DataFrame([
    {"split": c["split"], "season": c["season"], **{t: t in c["tags"] for t in all_tags}}
    for c in clips
])

summary = meta.groupby(["split", "season"])[all_tags].sum()
summary.insert(0, "clips", meta.groupby(["split", "season"]).size())
summary["no_special"] = (~meta[all_tags].any(axis=1)).groupby([meta["split"], meta["season"]]).sum()
summary.loc[["train", "dev", "test"]]

clips  hbp  ibb  pitch_clock  runner_out  walk_award  no_special
split season                                                                  
train 2023    12852  660   95            0         169          80       11869
      2024    12795  748   92            1         143          57       11783
      2025     6379  386   63           99          74          30        5760
dev   2025     6416  385   58          109          69          39        5792
test  2026     8345  509   51           84          89          55        7586

In [10]:
# 정답 이벤트를 그대로 상태머신에 넣으면 상태가 100% 재현되는지 확인 (= 완벽한 음성인식을 가정한 상한선)
def replay(events, start_outs=0):
    balls, strikes, outs = 0, 0, start_outs
    states = []
    for e in events:
        if "pa_end" in e:
            balls, strikes, outs = 0, 0, e["o"]
        elif e["type"] == "auto_ball":
            balls += 1
        elif e["type"] == "auto_strike":
            strikes += 1
        elif e["type"] == "pitch":
            balls, strikes, _ = apply_pitch(balls, strikes, e["result"])
        states.append((balls, strikes, outs))
    return states


wrong = 0
total = 0
for c in clips:
    pred = replay(c["events"])
    gold = [(e["b"], e["s"], e["o"]) for e in c["events"]]
    wrong += sum(p != g for p, g in zip(pred, gold))
    total += len(gold)

print(f"이벤트 {total}개 중 불일치 {wrong}개")

이벤트 806312개 중 불일치 0개


In [19]:
# 예시 하나 확인 + Drive에 저장 (한 번 저장해두면 06 등에서는 03을 다시 돌릴 필요 없음)
sample = next(c for c in clips if "pitch_clock" in c["tags"])
print(json.dumps({**sample, "events": sample["events"][:8]}, ensure_ascii=False, indent=1))

with open(DRIVE / "labels_all.jsonl", "w", encoding="utf-8") as f:
    for c in clips:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
print("저장:", DRIVE / "labels_all.jsonl", len(clips), "클립")

{
 "clip_id": "20240504OBLG02024_06T",
 "split": "train",
 "season": 2024,
 "game_pk": "20240504OBLG02024",
 "inning": 6,
 "half": "top",
 "away": "OB",
 "home": "LG",
 "score": {
  "away": 3,
  "home": 2
 },
 "tags": [
  "pitch_clock"
 ],
 "n_pitches": 18,
 "events": [
  {
   "type": "pitch",
   "ab": 43,
   "pitcher": "손주영",
   "batter": "양의지",
   "pitch": 1,
   "result": "T",
   "pitch_name": "커브",
   "speed": 119,
   "b": 0,
   "s": 1,
   "o": 0
  },
  {
   "type": "pitch",
   "ab": 43,
   "pitcher": "손주영",
   "batter": "양의지",
   "pitch": 2,
   "result": "B",
   "pitch_name": "슬라이더",
   "speed": 130,
   "b": 1,
   "s": 1,
   "o": 0
  },
  {
   "type": "pitch",
   "ab": 43,
   "pitcher": "손주영",
   "batter": "양의지",
   "pitch": 3,
   "result": "T",
   "pitch_name": "슬라이더",
   "speed": 129,
   "b": 1,
   "s": 2,
   "o": 0
  },
  {
   "type": "pitch",
   "ab": 43,
   "pitcher": "손주영",
   "batter": "양의지",
   "pitch": 4,
   "result": "B",
   "pitch_name": "슬라이더",
   "speed": 131,
   "b": 